In [ ]:
%pip install requests beautifulsoup4

In [ ]:
import time
import requests
from bs4 import BeautifulSoup

Paso 1: Parsear el archivo HTML local y extraer los datos y enlaces
Este código lee tu archivo local agroads_cordoba_lotes.html, extrae todos los textos del listado y prepara la lista de links para el crawler:

In [ ]:
import requests

# URL de la sección que queremos descargar
url = "https://www.agroads.com.ar/seccion.asp?prov=8&subcat=36"
headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    )
}

print(f"Descargando HTML de: {url}...")
response = requests.get(url, headers=headers)

if response.status_code == 200:
    with open("agroads_cordoba_lotes.html", "w", encoding="utf-8") as f:
        f.write(response.text)
    print("¡Archivo 'agroads_cordoba_lotes.html' guardado con éxito!")
else:
    print(f"Error al descargar la página. Código de estado: {response.status_code}")

Descargando HTML de: https://www.agroads.com.ar/seccion.asp?prov=8&subcat=36...
¡Archivo 'agroads_cordoba_lotes.html' guardado con éxito!


In [ ]:
import csv
import time
from bs4 import BeautifulSoup
import requests

# 1. Cargar y parsear el archivo HTML local
with open("agroads_cordoba_lotes.html", "r", encoding="utf-8", errors="ignore") as f:
    soup = BeautifulSoup(f, "html.parser")

items = soup.select(".item.anuncio")
print(f"Total de publicaciones encontradas en el HTML: {len(items)}")

publicaciones = []

for item in items:
    # Atributos del tag contenedor
    anuncio_id = item.get("data-id")

    # Título
    titulo_tag = item.select_one(".anuncio_title")
    titulo = titulo_tag.get_text(strip=True) if titulo_tag else ""

    # Aptitud / Detalles
    detalles_tag = item.select_one(".anuncio_details")
    aptitud = detalles_tag.get_text(strip=True) if detalles_tag else ""

    # Ubicación / Localidad
    ubicacion_tag = item.select_one(".anuncio-distancia span")
    ubicacion = ubicacion_tag.get_text(strip=True) if ubicacion_tag else ""

    # Vendedor
    usuario_tag = item.select_one(".anuncio_username span")
    vendedor = usuario_tag.get_text(strip=True) if usuario_tag else ""

    # Precio
    precio_tag = item.select_one(".anuncio_price")
    precio_texto = precio_tag.get_text(strip=True) if precio_tag else ""

    precio_wrapper = item.select_one(".precio_wrapper")
    precio_valor = precio_wrapper.get("data-price") if precio_wrapper else ""
    moneda = precio_wrapper.get("data-currency") if precio_wrapper else ""

    # Condiciones comerciales (puede haber más de una)
    condiciones = [
        c.get_text(strip=True) for c in item.select(".sdp_main_condition-text")
    ]
    condiciones_str = " | ".join(condiciones)

    # Link al detalle
    link_tag = item.find("a")
    href = link_tag.get("href") if link_tag else ""
    if href.startswith("/"):
        link_completo = f"https://www.agroads.com.ar{href}"
    elif href.startswith("http"):
        link_completo = href
    else:
        link_completo = f"https://www.agroads.com.ar/{href}" if href else ""

    publicaciones.append(
        {
            "id": anuncio_id,
            "titulo": titulo,
            "aptitud": aptitud,
            "ubicacion": ubicacion,
            "vendedor": vendedor,
            "precio_texto": precio_texto,
            "precio_valor": precio_valor,
            "moneda": moneda,
            "condiciones": condiciones_str,
            "link": link_completo,
        }
    )

print(f"Ejemplo del primer anuncio procesado:\n{publicaciones[0]}")

Total de publicaciones encontradas en el HTML: 60
Ejemplo del primer anuncio procesado:
{'id': '1092607', 'titulo': 'Alquilo 310 Has Agrícolas Soja/maiz el Fortin Córdoba', 'aptitud': 'Aptitud: Agrícola', 'ubicacion': 'El Fortín', 'vendedor': 'Monte Maiz Maquinarias', 'precio_texto': '$ Consultar', 'precio_valor': '0', 'moneda': '$', 'condiciones': '', 'link': 'https://www.agroads.com.ar/detalle.asp?clasi=1092607'}


### Proceso de Escaneo de Múltiples Páginas (Páginas 1 a 7)

A continuación, definimos el bloque para recorrer de la página 1 a la 7, descargar su contenido, extraer todos los anuncios de cada listado y consolidar una lista única de publicaciones.

In [ ]:
import time
import requests
from bs4 import BeautifulSoup

base_url = "https://www.agroads.com.ar/seccion.asp?prov=8&subcat=36"
headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    )
}

publicaciones_totales = []

for pag in range(1, 8):
    # Para la primera página usamos la URL base limpia, para las demás añadimos '&pag=X'
    if pag == 1:
        url = base_url
    else:
        url = f"{base_url}&pag={pag}"

    print(f"Descargando y procesando listado: {url}...")

    try:
        response = requests.get(url, headers=headers, timeout=15)
        if response.status_code == 200:
            soup = BeautifulSoup(response.text, "html.parser")
            items = soup.select(".item.anuncio")
            print(f"  -> Encontrados {len(items)} anuncios en la página {pag}")

            for item in items:
                anuncio_id = item.get("data-id")

                # Título
                titulo_tag = item.select_one(".anuncio_title")
                titulo = titulo_tag.get_text(strip=True) if titulo_tag else ""

                # Detalles
                detalles_tag = item.select_one(".anuncio_details")
                aptitud = detalles_tag.get_text(strip=True) if detalles_tag else ""

                # Ubicación
                ubicacion_tag = item.select_one(".anuncio-distancia span")
                ubicacion = ubicacion_tag.get_text(strip=True) if ubicacion_tag else ""

                # Vendedor
                usuario_tag = item.select_one(".anuncio_username span")
                vendedor = usuario_tag.get_text(strip=True) if usuario_tag else ""

                # Precio
                precio_tag = item.select_one(".anuncio_price")
                precio_texto = precio_tag.get_text(strip=True) if precio_tag else ""

                precio_wrapper = item.select_one(".precio_wrapper")
                precio_valor = precio_wrapper.get("data-price") if precio_wrapper else ""
                moneda = precio_wrapper.get("data-currency") if precio_wrapper else ""

                # Condiciones
                condiciones = [c.get_text(strip=True) for c in item.select(".sdp_main_condition-text")]
                condiciones_str = " | ".join(condiciones)

                # Enlace
                link_tag = item.find("a")
                href = link_tag.get("href") if link_tag else ""
                if href.startswith("/"):
                    link_completo = f"https://www.agroads.com.ar{href}"
                elif href.startswith("http"):
                    link_completo = href
                else:
                    link_completo = f"https://www.agroads.com.ar/{href}" if href else ""

                publicaciones_totales.append({
                    "id": anuncio_id,
                    "titulo": titulo,
                    "aptitud": aptitud,
                    "ubicacion": ubicacion,
                    "vendedor": vendedor,
                    "precio_texto": precio_texto,
                    "precio_valor": precio_valor,
                    "moneda": moneda,
                    "condiciones": condiciones_str,
                    "link": link_completo
                })
        else:
            print(f"  -> Error en página {pag}: código de estado {response.status_code}")

    except Exception as e:
        print(f"  -> Error al procesar la página {pag}: {e}")

    # Pausa de cortesía entre descargas de páginas de listado
    time.sleep(2)

print(f"\nProceso de listados completado. Total de anuncios recolectados: {len(publicaciones_totales)}")

Descargando y procesando listado: https://www.agroads.com.ar/seccion.asp?prov=8&subcat=36...
  -> Encontrados 60 anuncios en la página 1
Descargando y procesando listado: https://www.agroads.com.ar/seccion.asp?prov=8&subcat=36&pag=2...
  -> Encontrados 60 anuncios en la página 2
Descargando y procesando listado: https://www.agroads.com.ar/seccion.asp?prov=8&subcat=36&pag=3...
  -> Encontrados 60 anuncios en la página 3
Descargando y procesando listado: https://www.agroads.com.ar/seccion.asp?prov=8&subcat=36&pag=4...
  -> Encontrados 60 anuncios en la página 4
Descargando y procesando listado: https://www.agroads.com.ar/seccion.asp?prov=8&subcat=36&pag=5...
  -> Encontrados 60 anuncios en la página 5
Descargando y procesando listado: https://www.agroads.com.ar/seccion.asp?prov=8&subcat=36&pag=6...
  -> Encontrados 60 anuncios en la página 6
Descargando y procesando listado: https://www.agroads.com.ar/seccion.asp?prov=8&subcat=36&pag=7...
  -> Error en página 7: código de estado 202

Pro

Una vez consolidada la lista global, podemos re-asignar la variable `publicaciones` para que las celdas posteriores del crawler y exportación a CSV utilicen esta nueva base de datos completa de 7 páginas.

In [ ]:
# Reemplazamos la variable previa para que continúe con el flujo actual
publicaciones = publicaciones_totales
print(f"Variable 'publicaciones' actualizada con {len(publicaciones)} elementos para el crawler.")

Variable 'publicaciones' actualizada con 360 elementos para el crawler.


Paso 2: Crawler para recorrer link por link y extraer la información detallada
En la página individual (detalle.asp?clasi=...)
Ficha Técnica: Superficie, Transacción, Aptitud, Instalaciones.
Descripción extendida: Texto completo del anuncio.

In [ ]:
# Cabeceras completas simulando un navegador real
headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    ),
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "es-ES,es;q=0.9,en;q=0.8",
}

session = requests.Session()
session.headers.update(headers)

datos_completos = []

# Puedes probar con los primeros 5 cambiando a: publicaciones[:5]
for index, pub in enumerate(publicaciones, 1):
    url = pub["link"]
    print(f"[{index}/{len(publicaciones)}] Scrapeando: {url}")

    detalle_info = {
        **pub,  # Incluye los datos ya extraídos del listado
        "superficie_ha": "",
        "transaccion": "",
        "instalaciones": "",
        "descripcion_completa": "",
    }

    if not url:
        datos_completos.append(detalle_info)
        continue

    try:
        resp = session.get(url, timeout=12)

        if resp.status_code == 200:
            soup_detalle = BeautifulSoup(resp.text, "html.parser")

            # 1. Extraer pares de Ficha Técnica (<dt> Etiqueta : <dd> Valor)
            for dl in soup_detalle.find_all("dl", class_="grupo"):
                dts = [dt.get_text(strip=True) for dt in dl.find_all("dt")]
                dds = [dd.get_text(strip=True) for dd in dl.find_all("dd")]

                for etiqueta, valor in zip(dts, dds):
                    etiqueta_clean = etiqueta.lower()
                    if "superficie" in etiqueta_clean:
                        detalle_info["superficie_ha"] = valor
                    elif "transacci" in etiqueta_clean:
                        detalle_info["transaccion"] = valor
                    elif "instalacion" in etiqueta_clean:
                        detalle_info["instalaciones"] = valor

            # 2. Extraer descripción completa
            desc_elem = soup_detalle.find(
                "dl", class_=lambda c: c and "descripcion" in c
            )
            if desc_elem:
                dd_desc = desc_elem.find("dd")
                if dd_desc:
                    detalle_info["descripcion_completa"] = dd_desc.get_text(
                        " ", strip=True
                    )

        else:
            print(f"   -> Advertencia: Código de estado {resp.status_code}")

    except Exception as e:
        print(f"   -> Error al consultar {url}: {e}")

    datos_completos.append(detalle_info)

    # Pausa de 2.5 segundos para no ser bloqueado
    time.sleep(2.5)

print("\n¡Extracción finalizada con éxito!")

[1/360] Scrapeando: https://www.agroads.com.ar/detalle.asp?clasi=1092607
   -> Advertencia: Código de estado 202
[2/360] Scrapeando: https://www.agroads.com.ar/detalle.asp?clasi=1052572
   -> Advertencia: Código de estado 202
[3/360] Scrapeando: https://www.agroads.com.ar/detalle.asp?clasi=1091956
   -> Advertencia: Código de estado 202
[4/360] Scrapeando: https://www.agroads.com.ar/detalle.asp?clasi=1024403
   -> Advertencia: Código de estado 202
[5/360] Scrapeando: https://www.agroads.com.ar/detalle.asp?clasi=1090947
   -> Advertencia: Código de estado 202
[6/360] Scrapeando: https://www.agroads.com.ar/detalle.asp?clasi=1090780
   -> Advertencia: Código de estado 202
[7/360] Scrapeando: https://www.agroads.com.ar/detalle.asp?clasi=1090452
   -> Advertencia: Código de estado 202
[8/360] Scrapeando: https://www.agroads.com.ar/detalle.asp?clasi=1068709
   -> Advertencia: Código de estado 202
[9/360] Scrapeando: https://www.agroads.com.ar/detalle.asp?clasi=1085981
   -> Advertencia: Códi

In [ ]:
# Guardar en CSV compatible con Excel y UTF-8
archivo_salida = "anuncios_agroads_cordoba.csv"

if datos_completos:
    columnas = datos_completos[0].keys()
    with open(archivo_salida, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=columnas)
        writer.writeheader()
        writer.writerows(datos_completos)

    print(
        f"Se guardaron {len(datos_completos)} registros en '{archivo_salida}'."
    )

Se guardaron 60 registros en 'anuncios_agroads_cordoba.csv'.
